# Local LLM benchmark: GSM8K / AG News + CodeCarbon + EcoLogits
Before each session/ after reboot: `sudo chmod -R a+r /sys/class/powercap/*`
Ollama must be running. Choose task in config cell, then run top to bottom.

In [20]:
# Config
import json, re, time, random
from pathlib import Path
from dataclasses import asdict
from datetime import datetime, timedelta

import requests
import pandas as pd
from datasets import load_dataset
from codecarbon import OfflineEmissionsTracker
from ecologits.tracers.utils import llm_impacts

TASK = "agnews"                     # "gsm8k" or "agnews"
MODEL = "ministral-3:3b"           # Ollama tag
OLLAMA_URL = "http://localhost:11434/api/generate"
N_ITEMS = 100
SEED = 42
COUNTRY = "DEU"                    # only affects CO2, not energy
OUT_DIR = Path("results")
OUT_DIR.mkdir(exist_ok=True)
TAG = f"{TASK}_{MODEL.replace(':', '_')}"

# EcoLogits settings for the same model on datacentre hardware
ECO_PROVIDER = "mistralai"
ECO_MODEL = "ministral-3b-latest"
ECO_ZONE = "DEU"

In [ ]:
# Task definitions: loading, prompt, answer extraction, scoring
NUM_RE = re.compile(r"-?\d[\d,]*\.?\d*")
AG_LABELS = ["World", "Sports", "Business", "Sci/Tech"]

def gsm8k_extract(text):
    # Prefer the "####" marker, fall back to the last number in the text
    if "####" in text:
        m = NUM_RE.search(text.split("####")[-1])
        if m:
            return m.group().replace(",", "").rstrip(".")
    nums = NUM_RE.findall(text)
    return nums[-1].replace(",", "").rstrip(".") if nums else None

def gsm8k_correct(pred, gold):
    try:
        return pred is not None and abs(float(pred) - float(gold)) < 1e-6
    except ValueError:
        return False

def agnews_extract(text):
    # Take the first label mentioned in the answer
    t = text.lower()
    aliases = {
        "World": ["world"],
        "Sports": ["sports", "sport"],
        "Business": ["business"],
        "Sci/Tech": ["sci/tech", "sci-tech", "science", "technology", "tech"],
    }
    hits = []
    for label, words in aliases.items():
        pos = [t.find(w) for w in words if t.find(w) >= 0]
        if pos:
            hits.append((min(pos), label))
    return min(hits)[1] if hits else None

TASKS = {
    "gsm8k": {
        "load": lambda: load_dataset("openai/gsm8k", "main", split="test"),
        "prompt": (
            "Solve the following math problem step by step. "
            "At the end, write the final answer as a single number on its own line "
            "in the format: #### <number>\n\nProblem: {question}"
        ),
        "question": lambda item: item["question"],
        "gold": lambda item: item["answer"].split("####")[-1].strip().replace(",", ""),
        "extract": gsm8k_extract,
        "correct": gsm8k_correct,
        "max_tokens": 512,
    },
    "agnews": {
        "load": lambda: load_dataset("fancyzhx/ag_news", split="test"),
        "prompt": (
            "Classify the following news article into exactly one of these categories: "
            "World, Sports, Business, Sci/Tech. "
            "Answer with the category name only.\n\nArticle: {question}"
        ),
        "question": lambda item: item["text"],
        "gold": lambda item: AG_LABELS[item["label"]],
        "extract": agnews_extract,
        "correct": lambda pred, gold: pred == gold,
        "max_tokens": 10,
    },
}

T = TASKS[TASK]
MAX_TOKENS = T["max_tokens"]
PROMPT = T["prompt"]

In [ ]:
# Fixed random sample, saved per task so API run uses same items
ds = T["load"]()
ids_path = OUT_DIR / f"item_ids_{TASK}.json"
if ids_path.exists():
    idx = json.loads(ids_path.read_text())
else:
    random.seed(SEED)
    idx = sorted(random.sample(range(len(ds)), N_ITEMS))
    ids_path.write_text(json.dumps(idx))
items = [ds[i] | {"idx": i} for i in idx]
print(len(items), T["question"](items[0])[:200])

100 Vodafone hires Citi for Cesky bid (TheDeal.com) TheDeal.com - The U.K. mobile giant wants to find a way to disentagle the Czech wireless and fixed-line businesses.


In [ ]:
# Call Ollama (non-streaming); returns text plus token counts && timings
def ask(question):
    payload = {
        "model": MODEL,
        "prompt": PROMPT.format(question=question),
        "stream": False,
        "options": {"temperature": 0, "seed": SEED, "num_predict": MAX_TOKENS},
    }
    t0 = time.perf_counter()
    r = requests.post(OLLAMA_URL, json=payload, timeout=600)
    r.raise_for_status()
    latency = time.perf_counter() - t0
    d = r.json()
    return {
        "output": d["response"],
        "input_tokens": d.get("prompt_eval_count", 0),
        "output_tokens": d.get("eval_count", 0),
        "latency_s": latency,
        "gen_s": d.get("eval_duration", 0) / 1e9,
    }

# Warm-up: loads the model into memory so loading is !measured
print(ask("What is 2 + 2?")["output"][:200])

World


In [ ]:
# Idle baseline: machine power with Ollama loaded but !generating
IDLE_SECONDS = 60
tracker = OfflineEmissionsTracker(
    project_name="idle", country_iso_code=COUNTRY,
    measure_power_secs=1, tracking_mode="machine",
    output_dir=str(OUT_DIR), log_level="warning",
)
tracker.start()
time.sleep(IDLE_SECONDS)
tracker.stop()
idle = tracker.final_emissions_data
idle_power_w = idle.energy_consumed * 3.6e6 / idle.duration
print(f"Idle power: {idle_power_w:.1f} W over {idle.duration:.0f} s")

[codecarbon WARNING @ 15:54:14] Multiple instances of codecarbon are allowed to run at the same time.


Idle power: 41.8 W over 60 s


In [ ]:
# Main run: one tracker around whole benchmark
rows = []
tracker = OfflineEmissionsTracker(
    project_name=TAG, country_iso_code=COUNTRY,
    measure_power_secs=1, tracking_mode="machine",
    output_dir=str(OUT_DIR), log_level="warning",
)
tracker.start()
for k, item in enumerate(items):
    res = ask(T["question"](item))
    gold = T["gold"](item)
    pred = T["extract"](res["output"])
    rows.append({"idx": item["idx"], "gold": gold, "pred": pred,
                 "correct": T["correct"](pred, gold), **res})
    if k % 10 == 0:
        print(f"{k}/{len(items)} acc so far: {sum(r['correct'] for r in rows)/len(rows):.2f}")
tracker.stop()
run = tracker.final_emissions_data
df = pd.DataFrame(rows)
print(f"Accuracy: {df.correct.mean():.3f}")

[codecarbon WARNING @ 15:55:36] Multiple instances of codecarbon are allowed to run at the same time.


0/100 acc so far: 0.00
10/100 acc so far: 0.55
20/100 acc so far: 0.48
30/100 acc so far: 0.52
40/100 acc so far: 0.51
50/100 acc so far: 0.49
60/100 acc so far: 0.48
70/100 acc so far: 0.45
80/100 acc so far: 0.44
90/100 acc so far: 0.45
Accuracy: 0.460


In [26]:
# EcoLogits estimate per item (usage-phase energy, comparable to CodeCarbon)
def eco(row):
    imp = llm_impacts(
        provider=ECO_PROVIDER, model_name=ECO_MODEL,
        output_token_count=int(row.output_tokens),
        request_latency=float(row.latency_s),
        electricity_mix_zone=ECO_ZONE,
    )
    if imp is None:
        return pd.Series({"eco_kwh_min": None, "eco_kwh_max": None})
    e = imp.usage.energy.value
    lo, hi = (e.min, e.max) if hasattr(e, "min") else (e, e)
    return pd.Series({"eco_kwh_min": lo, "eco_kwh_max": hi})

df[["eco_kwh_min", "eco_kwh_max"]] = df.apply(eco, axis=1)

In [27]:
# Summary + full export (all CodeCarbon fields, time info, derived metrics)
idle_kwh = idle_power_w * run.duration / 3.6e6
net_kwh = max(run.energy_consumed - idle_kwh, 0)
df["cc_kwh"] = net_kwh * df.latency_s / df.latency_s.sum()

end = datetime.fromisoformat(str(run.timestamp))
start = end - timedelta(seconds=run.duration)
n = len(df)
n_correct = int(df.correct.sum())
out_tokens = max(int(df.output_tokens.sum()), 1)
gross_wh = run.energy_consumed * 1000
net_wh = net_kwh * 1000

summary = {
    "task": TASK, "model": MODEL, "setting": "local",
    "run_start": start.isoformat(), "run_end": end.isoformat(),
    "start_hour": start.hour, "weekday": start.strftime("%A"),
    "n": n, "accuracy": df.correct.mean(), "n_correct": n_correct,
    "mean_input_tokens": df.input_tokens.mean(),
    "mean_output_tokens": df.output_tokens.mean(),
    "mean_latency_s": df.latency_s.mean(),
    "tokens_per_s": out_tokens / df.gen_s.sum() if df.gen_s.sum() > 0 else None,
    "idle_power_w": idle_power_w,
    "avg_power_w": run.energy_consumed * 3.6e6 / run.duration,
    "total_kwh": run.energy_consumed, "idle_kwh": idle_kwh, "net_kwh": net_kwh,
    "cpu_kwh": run.cpu_energy, "gpu_kwh": run.gpu_energy, "ram_kwh": run.ram_energy,
    "gross_wh_per_query": gross_wh / n, "net_wh_per_query": net_wh / n,
    "gross_wh_per_correct": gross_wh / max(n_correct, 1),
    "net_wh_per_correct": net_wh / max(n_correct, 1),
    "gross_wh_per_1k_tokens": gross_wh / out_tokens * 1000,
    "net_wh_per_1k_tokens": net_wh / out_tokens * 1000,
    "idle_share": idle_kwh / run.energy_consumed,
    "cc_g_co2_per_query": run.emissions * 1000 / n,
    "eco_wh_per_query_min": df.eco_kwh_min.mean() * 1000,
    "eco_wh_per_query_max": df.eco_kwh_max.mean() * 1000,
}
stamp = f"{start:%Y%m%d_%H%M}"
df.to_csv(OUT_DIR / f"items_{TAG}_{stamp}.csv", index=False)

details = summary | {f"cc_{k}": v for k, v in asdict(run).items()} \
                  | {f"idle_{k}": v for k, v in asdict(idle).items()}
pd.DataFrame([details]).to_csv(OUT_DIR / f"run_details_{TAG}_{stamp}.csv", index=False)

path = OUT_DIR / "runs_local.csv"
pd.DataFrame([summary]).to_csv(path, mode="a", header=not path.exists(), index=False)
pd.Series(summary)

task                                          agnews
model                                 ministral-3:3b
setting                                        local
run_start                 2026-11-18T15:55:35.527470
run_end                          2026-11-18T15:58:47
start_hour                                        15
weekday                                    Wednesday
n                                                100
accuracy                                        0.46
n_correct                                         46
mean_input_tokens                             666.42
mean_output_tokens                              2.16
mean_latency_s                              1.914411
tokens_per_s                                15.07409
idle_power_w                                 41.7573
avg_power_w                                45.677828
total_kwh                                   0.002429
idle_kwh                                    0.002221
net_kwh                                     0.

In [17]:
print(df.pred.value_counts(dropna=False))
print(pd.crosstab(df.gold, df.pred.fillna("None")))
df[~df.correct][["gold", "pred", "output"]].head(15)

pred
World       66
Business    22
Sports      10
Sci/Tech     2
Name: count, dtype: int64
pred      Business  Sci/Tech  Sports  World
gold                                       
Business        16         0       0     14
Sci/Tech         4         2       1     19
Sports           2         0       9     12
World            0         0       0     21


,gold,pred,output
0,Sci/Tech,Business,Business
1,Sports,World,World
4,Sports,World,World
6,Sci/Tech,World,World
8,Sci/Tech,World,World
10,Business,World,World
11,Sports,World,World
12,Sports,World,World
16,Sci/Tech,World,World
17,Sci/Tech,World,World


In [18]:
df[~df.correct][["gold", "pred", "output"]].head(10)

,gold,pred,output
0,Sci/Tech,Business,Business
1,Sports,World,World
4,Sports,World,World
6,Sci/Tech,World,World
8,Sci/Tech,World,World
10,Business,World,World
11,Sports,World,World
12,Sports,World,World
16,Sci/Tech,World,World
17,Sci/Tech,World,World


In [19]:
test_ids = df[~df.correct].idx.head(10).tolist()
for i in test_ids:
    item = ds[i]
    payload = {"model": MODEL, "prompt": PROMPT.format(question=item["text"]),
               "system": "", "stream": False,
               "options": {"temperature": 0, "seed": SEED, "num_predict": MAX_TOKENS}}
    d = requests.post(OLLAMA_URL, json=payload).json()
    print(AG_LABELS[item["label"]], "|", d["response"].strip(), "|", d["prompt_eval_count"])

Sci/Tech | Business | 629
Sports | World | 634
Sports | World | 639
Sci/Tech | World | 622
Sci/Tech | World | 641
Business | World | 640
Sports | World | 640
Sports | World | 626
Sci/Tech | World | 645
Sci/Tech | Sports | 629
